# Task 3 — Biến đổi hình học ảnh

Thực hiện translation, rotation, scaling, affine và projective (homography) trên 5 ảnh JPG trong `input/`. `sample.png` đã được loại bỏ. Phần hiển thị dùng cùng 5 phép biến đổi cho mỗi ảnh để dễ so sánh.

Affine dùng 3 cặp điểm nên bảo toàn đường thẳng và tính song song. Homography dùng 4 cặp điểm, vẫn bảo toàn đường thẳng nhưng có thể làm các đường song song hội tụ, mô phỏng phối cảnh. Trong ví dụ này, các điểm đích được chọn thủ công để minh họa, không phải kết quả hiệu chỉnh một cảnh thực.

Ảnh tải từ Unsplash: [architecture](https://images.unsplash.com/photo-1511818966892-d7d671e672a2), [landscape](https://images.unsplash.com/photo-1500530855697-b586d89ba3ee), [mountains](https://images.unsplash.com/photo-1470770841072-f978cf4d019e), [ocean](https://images.unsplash.com/photo-1518837695005-2083093ee35b), [sunset](https://images.unsplash.com/photo-1470252649378-9c29740c9fa8).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

input_dir = Path('input')
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

image_paths = sorted(input_dir.glob('*.jpg'))
images = {path.name: Image.open(path).convert('RGB') for path in image_paths}
print(f'Đã đọc {len(images)} ảnh JPG: {", ".join(images)}')

In [ ]:
def warp(image, matrix):
    width, height = image.size
    if matrix.shape == (2, 3):
        inverse = np.linalg.inv(np.vstack([matrix, [0, 0, 1]]))
        transform = Image.Transform.AFFINE
        coefficients = inverse[:2].ravel()
    else:
        inverse = np.linalg.inv(matrix)
        inverse /= inverse[2, 2]
        transform = Image.Transform.PERSPECTIVE
        coefficients = inverse.ravel()[:8]

    return image.transform(
        (width, height), transform, tuple(coefficients),
        resample=Image.Resampling.BILINEAR, fillcolor=(0, 0, 0)
    )


def transform_image(image):
    width, height = image.size
    center_x, center_y = width / 2, height / 2

    # Dịch ảnh 10% chiều rộng sang phải và 8% chiều cao xuống dưới.
    translation = np.array([[1, 0, .10 * width], [0, 1, .08 * height]])

    # Quay 12 độ ngược chiều kim đồng hồ quanh tâm ảnh (trục y hướng xuống).
    angle = np.deg2rad(12)
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    rotation = np.array([
        [cos_a, sin_a, (1 - cos_a) * center_x - sin_a * center_y],
        [-sin_a, cos_a, sin_a * center_x + (1 - cos_a) * center_y],
    ])

    # Thu ảnh còn 78%, giữ tâm tại chỗ.
    scale = .78
    scaling = np.array([
        [scale, 0, (1 - scale) * center_x],
        [0, scale, (1 - scale) * center_y],
    ])

    # Ba cặp điểm xác định affine; cạnh song song vẫn song song.
    source_triangle = np.float64([[0, 0], [width - 1, 0], [0, height - 1]])
    target_triangle = np.float64([
        [.10 * width, .08 * height],
        [.88 * width, .12 * height],
        [.16 * width, .88 * height],
    ])
    affine = np.linalg.solve(
        np.column_stack([source_triangle, np.ones(3)]), target_triangle
    ).T

    # Bốn góc xác định homography; tạo hình thang để thấy hiệu ứng phối cảnh.
    source_corners = np.float64([
        [0, 0], [width - 1, 0], [0, height - 1], [width - 1, height - 1]
    ])
    target_corners = np.float64([
        [.20 * width, .06 * height], [.80 * width, .06 * height],
        [.05 * width, .94 * height], [.95 * width, .94 * height],
    ])
    rows = []
    for (x, y), (u, v) in zip(source_corners, target_corners):
        rows.extend([
            [x, y, 1, 0, 0, 0, -u * x, -u * y],
            [0, 0, 0, x, y, 1, -v * x, -v * y],
        ])
    homography = np.append(
        np.linalg.solve(rows, target_corners.ravel()), 1
    ).reshape(3, 3)

    matrices = {
        'Original': None,
        'Translation': translation,
        'Rotation 12°': rotation,
        'Scaling 0.78×': scaling,
        'Affine': affine,
        'Projective': homography,
    }
    return {
        name: image if matrix is None else warp(image, matrix)
        for name, matrix in matrices.items()
    }


results = {name: transform_image(image) for name, image in images.items()}

In [ ]:
titles = list(next(iter(results.values())))
fig, axes = plt.subplots(len(results), len(titles), figsize=(18, 3.6 * len(results)))
for row, (filename, transformed) in enumerate(results.items()):
    for column, title in enumerate(titles):
        axes[row, column].imshow(transformed[title])
        axes[row, column].set_title(f'{filename}\n{title}')
        axes[row, column].axis('off')
plt.tight_layout()
plt.show()

# Lưu kết quả affine và homography của từng ảnh JPG.
for filename, transformed in results.items():
    stem = Path(filename).stem
    transformed['Affine'].save(output_dir / f'{stem}_affine.jpg')
    transformed['Projective'].save(output_dir / f'{stem}_projective.jpg')
print(f'Đã lưu 2 kết quả cho mỗi ảnh vào {output_dir.resolve()}')

## Đánh giá

Translation chỉ đổi vị trí; ảnh có thể bị cắt vì canvas cố định. Rotation giữ tỷ lệ nhưng tạo vùng đen ở các góc. Scaling thu ảnh quanh tâm nên lộ viền đen. Affine nghiêng/biến dạng ảnh nhưng các cặp cạnh song song vẫn song song. Homography cho cạnh trên hẹp hơn cạnh dưới, thể hiện phối cảnh và làm thay đổi tính song song. Các điểm đích được chọn thủ công nên đây là demo hình học, không phải phép nắn phối cảnh đo từ vật thể thật. Nội suy song tuyến tính làm mượt pixel khi lấy mẫu lại.